# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


## Cumulative midterm checkpoint (human review)

Cumulative Checkpoint Responses:

Lecture 01:

In task 1.2, I created a function called "profile" that stores a list of dictionaries storing the name, number of rows, number of columns, and number of duplicates for each file stored in "data_manifest.json".

Lecture 02:

In task 1.2, I appended each file in "data_manifest.json" to my "results" list using "results.append()". 

Lecture 03:

When calculating the mean across columns in task 4.3, I selected axis = 0 as my argument for mean(). Axis = 0 was used as the visit (baseline and week12) were represented by the columns and axis = 0 calculates the mean vertically by column (the means per visit which I am looking for).

Lecture 04: 

Using ipynb Pandas notebook, I converted a list called "results" into a DataFrame using pd.DataFrame() in task 1.2.

Lecture 05:

Task 2.3 checks to make sure that the visit date is formatted correctly as 'YYYY-MM-DD'. It counts the number of rows without the correct format and saves it in "issue_audit" with "date parse failures." 

Lecture 06:

Task 4.1 melts raw["a1c_long"] into a wide format that lists the baseline visits first, followed by week12 visits.


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
def profile(name, table):
    dict_1 = {"table": name, "rows": table.shape[0], "columns": table.shape[1], "exact_duplicate_rows": table.duplicated().sum()}
    return dict_1

results = list()
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

results = pd.DataFrame(results)
results.to_csv(PROFILE_PATH, index=False)



## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

Task 2.1 Response:

Patients:

As shown in task 3.1, the grain of the patients_clean table is that it represents the characteristics of each patient, including gender, birth_year, and site_code. The pkey for this table is patient_id. A many-to-one cardinality is expected when joining patients_clean with visits_clean in task 4.4 as visits_clean contains multiple visits from the same patient and matches the unique patient_id provided in patients_clean.


Visits: 

As shown in task 3.2, the grain of the visits_clean table is that it represents the details of the visit, including visit type (baseline or week12), visit_date, visit_time, and the parameters measured during the visit (e.g. sbp_mmhg and weight_kg). The key is the unique patient_id, visit pair, denoting different visit types for each patient. Stacking the visit types by source file formatted the table so that visits from northern sites were listed first, following by southern site visits. Keeping the last duplicate ensures that the most recent and updated measurements are kept for each patient and visit type pair when there are duplicate pairs. Many-to-one cardinalities are expected when left-merging visits_clean with other files in task 4.4 as patient_id and site_code values are repeated throughout the table, with the values being matched with the unique information provided in the patient_values, a1c_long, and site_code tables.

To clean the raw table, exact duplicates were removed, visit entries were mapped and renamed to "baseline" and "week12" to standardize our data, dates that didn't fit the YYYY-MM-DD format or were not a valid calendar day were marked as NaT, and sbp_mmhg measurements outside of the 70-250 range were removed to avoid potentially faulty measurements. Rechecks were done in task 3.2 to ensure individual entries were kept for each patient_id, visit pair. 


Sites:

As shown in task 4.2, the grain of the cleaned sites_current table is that it provides current site details including site_name and record_status. The key is the unique site_code. Prior to cleanup, the sites table contained duplicates of the site_code, as some sites were retired and replaced by another clinic. Keeping the current sites in the "sites_current" table created unique site codes for each current location, with no duplicates. Due to this change, a many-to-one cardinality is expected when joining sites_current with visits_clean in task 4.4 as visits_clean contains multiple visits from the same patient and matches the unique site_code provided in sites_current.  

The raw sites data contained data entries that violated the intended clean keys (e.g. "N01", "S02") as they contained hyphens, dashes, and whitespaces. These symbols and spaces were removed when cleaning my dataset. "sites_current" was also checked in task 4.2 to determine with the site codes were unique and could be joined using a "many-to-one" cardinality.


A1c_long join keys: 

In task 4.4, the join key for a1c_long is a pair, patient_id and visit. A pair was used for the join as a1c measurements varied depending on the patient_id and whether it was a baseline or week12 visit. Hence, the a1c_long and visits_clean joined on the two columns as they were both needed to retrieve the correct a1c measurement.

### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
# Creates a copy of raw["visits_north"] and designates the visit_north.csv file name as the source file.
raw_north = raw["visits_north"].copy(deep = True)
raw_north["source_file"] = "visits_north.csv"

# Renames wt_kg to weight_kg to a copy of raw["visits_south"] and designates the visit_south.csv file name as the source file.
raw_south = raw["visits_south"].copy(deep = True)
raw_south = raw_south.rename(columns = {"wt_kg": "weight_kg"})
raw_south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([raw_north, raw_south], ignore_index = True)

display(visits_raw)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.

# Copied patients and sites data into two separate variables. 
patients = Path("data") / "patients.parquet"
patients = pd.read_parquet(patients)
sites = Path("data") / "sites.csv"
sites = pd.read_csv(sites)

# Utilized my patients and sites data to calculate the issue counts in issue_audit.
issue_audit = list()
issue_audit.append(["patients: gender values that are not a code", sum(~(patients["gender"].str.strip().isin(["F", "M", "X"])))])
issue_audit.append(["patients: empty gender", (patients["gender"].str.strip()=="").sum()])
issue_audit.append(["patients: birth_year sentinel 9999", (patients["birth_year"].str.strip()=='9999').sum()])
issue_audit.append(["patients: site codes needing normalization", sum(~(patients["site_code"].str.strip().isin(sites["site_code"])))])
issue_audit.append(["sites: rows with a repeated site_code", sum(sites.duplicated(subset=["site_code"], keep=False))])
issue_audit.append(["visits: visit labels that are not baseline or week12", sum(~((visits_raw["visit"] == "baseline") | (visits_raw["visit"] == "week12")))])

sbps = pd.to_numeric(visits_raw["sbp_mmhg"])
issue_audit.append(["visits: sbp sentinel 999", sum(sbps == 999)])
issue_audit.append(["visits: sbp outside 70 and through 250", sum((sbps < 70)| ((sbps > 250) & ~(sbps == 999)))])
issue_audit.append(["visits: date parse failures", sum(~(visits_raw["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')))])
issue_audit.append(["sites: rows in repeated patient_id and visit pairs", sum(visits_raw.duplicated(subset=["patient_id", "visit"], keep=False))])
issue_audit = pd.DataFrame(issue_audit, columns = ["issue", "count"])

display(issue_audit)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
issue_audit.to_csv(AUDIT_PATH, index = False)

,issue,count
0,patients: gender values that are not a code,7026
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1918
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 and through 250,116
8,visits: date parse failures,77
9,sites: rows in repeated patient_id and visit p...,288


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_task3 = raw["patients"].copy(deep=True)
patients_task3 = patients_task3.drop_duplicates(subset = None, keep='first')
gender_counts = patients_task3["gender"].value_counts()
patients_task3["gender"] = patients_task3["gender"].str.strip().str.lower().map({"f": "F", "female": "F", "woman": "F", "m": "M", "male": "M", "man": "M",
                                                                       "x": "X", "nb": "X", "non-binary": "X"})
gender_counts = patients_task3["gender"].value_counts()
patients_task3["gender"] = patients_task3["gender"].astype("category")
patients_task3["birth_year"] = patients_task3["birth_year"].map(lambda x: np.nan if x == "9999" else x)
patients_task3["site_code"] = patients_task3["site_code"].str.strip().str.upper().str.replace("-", "").str.replace(" ", "")

patients_clean = patients_task3.reset_index(drop=True)
display(patients_clean)
display(patients_clean["gender"].value_counts())

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,NaN,N01
2,P00003,F,1970,N01
3,P00004,X,NaN,N01
4,P00005,M,1947,N01
...,...,...,...,...
29995,P29996,F,1967,S02
29996,P29997,F,1970,N01
29997,P29998,NaN,1967,N01
29998,P29999,M,1956,S02


gender
F    14413
M    13955
X      651
Name: count, dtype: int64

In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

display(patients_clean["gender"].cat.categories)

# Creates a DataFrame called summary which lists the 5 metrics required in task 3.1.
summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(), (patients_clean["site_code"] == "N01").sum(),
              (patients_clean["site_code"] == "S02").sum(), (patients_clean["site_code"] == "E04").sum()],
})

summary.to_csv(PATIENTS_SUMMARY_PATH, index = False)

# Creates a csv file called gender_counts which describes the number of patients and code for each category.
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [8]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.

# Completes steps 1-4 in task 3.2.
visits_task3 = visits_raw.copy(deep=True)
visits_task3 = visits_task3.drop_duplicates(subset = None, keep = "first")
visit_counts = visits_task3["visit"].value_counts()
visits_task3["visit"] = visits_task3["visit"].str.strip().str.lower().map({"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12"})
visit_counts = visits_task3["visit"].value_counts()
visits_task3["sbp_mmhg"] = pd.to_numeric(visits_task3["sbp_mmhg"], errors = "coerce")
visits_task3["sbp_mmhg"] = visits_task3["sbp_mmhg"].map(lambda x: np.nan if (x<70)|(x>250) else x)
visits_task3["weight_kg"] = pd.to_numeric(visits_task3["weight_kg"], errors = "coerce")

# Adjusts visit_date following step 5, task 3.2.
visit_date_match = visits_task3["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
visit_date_all = visits_task3["visit_date"]
visits_task3["visit_date"] = pd.to_datetime(visit_date_all.where(visit_date_match), format = '%Y-%m-%d', errors = 'coerce')
visits_clean = visits_task3
display(visits_clean)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132.0,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129.0,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148.0,67.5,visits_north.csv
3,P00003,week12,2026-04-07,10:00,137.0,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145.0,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57331,P30038,week12,2026-04-20,09:05,135.0,109.0,visits_south.csv
57332,P30039,baseline,2026-02-24,10:10,142.0,81.8,visits_south.csv
57333,P30039,week12,2026-05-15,10:05,138.0,80.3,visits_south.csv
57334,P30040,baseline,2026-02-04,14:40,144.0,121.0,visits_south.csv


In [9]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.
display(visits_clean[visits_clean.duplicated(subset = ["patient_id", "visit"], keep = False)])
visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset = ["patient_id", "visit"], keep = "last")

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145.0,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142.0,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153.0,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152.0,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121.0,72.3,visits_north.csv
...,...,...,...,...,...,...,...
55282,P28051,week12,2026-04-16,09:35,126.0,92.7,visits_south.csv
55472,P28232,baseline,2026-01-12,10:00,153.0,61.0,visits_south.csv
55473,P28232,baseline,2026-01-12,10:10,159.0,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151.0,74.8,visits_south.csv


In [10]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

# Sorts visits_clean by patient_id and visit, dropping visit_time.
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean.drop(columns = "visit_time")
display(visits_clean)
display(visits_clean["visit"].value_counts(dropna=False))

# Creates a DataFrame visits_summary with the seven metrics required for step 7.
visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits", "visit_date missing", "visit_date min", "visit_date max", "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max", "weight_kg missing", "weight_kg mean", "visits from visits_north.csv", "visits from visits_south.csv"],
    "value": [len(visits_clean), sum(visits_clean["visit"]=="baseline"), sum(visits_clean["visit"]=="week12"), visits_clean["visit_date"].isna().sum(), visits_clean["visit_date"].min(), visits_clean["visit_date"].max(),
              visits_clean["sbp_mmhg"].isna().sum(), visits_clean["sbp_mmhg"].min(), visits_clean["sbp_mmhg"].max(), visits_clean["weight_kg"].isna().sum(), visits_clean["weight_kg"].mean(), sum(visits_clean["source_file"]=="visits_north.csv"), sum(visits_clean["source_file"]=="visits_south.csv")],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index = False)

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,source_file
0,P00001,baseline,2026-03-06,146.0,52.4,visits_south.csv
1,P00001,week12,2026-05-27,129.0,50.2,visits_south.csv
2,P00002,baseline,2026-02-20,132.0,85.2,visits_north.csv
3,P00002,week12,2026-05-08,129.0,82.0,visits_north.csv
4,P00003,baseline,2026-01-16,148.0,67.5,visits_north.csv
...,...,...,...,...,...,...
57176,P30038,week12,2026-04-20,135.0,109.0,visits_south.csv
57177,P30039,baseline,2026-02-24,142.0,81.8,visits_south.csv
57178,P30039,week12,2026-05-15,138.0,80.3,visits_south.csv
57179,P30040,baseline,2026-02-04,144.0,121.0,visits_south.csv


visit
baseline    29599
week12      27582
Name: count, dtype: int64

### 3.3 Record the decisions

In [11]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = list()
decision_table.append(["patients", "exact duplicate rows", "dropped exact duplicates, keeping only the first one", "duplicates were dropped so that only one visit is recorded per row and avoiding confusion with writing the same visit multiple times"])
decision_table.append(["patients", "gender spellings that are not a code", "removed excess whitespace from gender data entries using str.strip(), lowercased my entries, and mapped my entries to display only M, F, or X", "stripping and lower-casing my entries simplifies the mapping process and standardizes my patient dataset so that there is only one letter for each gender"])
decision_table.append(["patients", "site codes needing normalization", "hyphens and dashes were removed from the site codes and letters were upper-cased", "these letter/symbol adjustments were made to ensure all site codes follow the same format and to ensure that each code is not written in multiple different formats, standardizing my patient dataset and making it easier to evaluate"])
decision_table.append(["patients", "birth_year sentinel 9999", "patients with birth years of 9999 had the birth_year rewritten as NaN", "the birth year was likely not recorded or is unknown"])
decision_table.append(["visits", "weight column named wt_kg in visits_south.csv", "the column name wt_kg was rewritten to weight_kg on the visits_raw DataFrame", "rewriting the column name and writing out the whole abbreviation provides clarity in my visits dataset and makes it easier for users to read and understand it"])
decision_table.append(["visits", "sbp sentinel 999 and readings outside 70 through 250", "sbp values outside of the 70-250mmHg range and 999 were marked as NaN", "these readings were likely faulty if performed on a healthy patient and would need to be repeated"])
decision_table.append(["visits", "impossible or loosely written visit dates", "visit dates without the YYYY-MM-DD format were marked as NaN", "writing the date using the same format ensures consistency and makes my visits dataset easier for users to read, understand, and extract data from"])
decision_table.append(["visits", "repeat patient_id and visit pairs", "visits_clean was sorted by visit time results in only one row of the patient_id, visit pair which makes the visit dataset easier to read; the last duplicate is also kept as tests likely had to be repeated to ensure accuracy"])
decision_table.append(["sites", "repeated site_code rows", "no further action taken at this time, besides reformatting the code names for consistency", "the exact clinic where the patients received care and when the duplicate clinic was retired are unknown"])
decision_table = pd.DataFrame(decision_table, columns=["table", "issue", "decision", "reason"])
decision_table.to_csv(DECISION_LOG_PATH, index=False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).

# Melts a1c_wide into a long form and maps the visit type to "baseline" and "week12" results. Rows with no results are dropped.
a1c_long = pd.melt(raw["a1c_wide"], id_vars = ["patient_id"], value_vars = ["a1c_baseline", "a1c_week12"], var_name = "visit", value_name = "a1c_pct")
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})
display(a1c_long.dtypes)
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

# Creates a DataFrame a1c_summary with the per-visit summary required for task 4.1.
a1c_summary = pd.DataFrame({"visit": ["baseline", "week12"], "results": [sum(a1c_long["visit"] == "baseline"), sum(a1c_long["visit"] == "week12")],
                            "a1c_pct_mean": [a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean()]})
a1c_summary.to_csv(A1C_SUMMARY_PATH,index=False)

patient_id     string
visit             str
a1c_pct       float64
dtype: object

### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
try:
    sites_merged = pd.merge(patients_clean, sites, on = "site_code", how = "left", validate = "many_to_one") 
except pd.errors.MergeError as x:
    print(f"Merge error: {x}")
display(sites[sites.duplicated(subset = ["site_code"], keep=False)])
sites_current = sites[sites["record_status"] == "current"]

Merge error: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

# Outer-merges visits_clean and a1c_long and evaluates by merge type (e.g. both, left only, right only).
outer_merge = pd.merge(visits_clean, a1c_long, on = ["patient_id", "visit"], how = "outer", indicator = True)
display(outer_merge["_merge"].value_counts())

# Creates a DataFrame called merge_audit with the seven self-checks in task 4.2.
merge_audit = pd.DataFrame({"check": ["visits with an a1c result", "visits without an a1c result", "A1c results without a visit", "visits whose patient is not on the registry", "registry patients with no visit", "registry patients whose site is not in the site list", "site list rows removed by the current-record rule"], 
                            "count": [sum((outer_merge["_merge"] == "both")), sum(outer_merge["_merge"] == "left_only"), sum(outer_merge["_merge"] == "right_only"),
                                      sum(~(outer_merge["patient_id"].isin(patients_clean["patient_id"]))), sum(~(patients_clean["patient_id"].isin(outer_merge["patient_id"]))), 
                                      sum(~patients_clean["site_code"].isin(sites["site_code"])), len(sites)-len(sites_current)]})
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)



_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]
sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].to_csv(SBP_WIDE_SAMPLE_PATH, index=False)

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.
# Creates complete pairs, a table comprised of patients that have both baseline and week12 sbp measurements.
sbp_wide_both = pd.DataFrame({"baseline": sbp_wide["baseline"], "week12": sbp_wide["week12"]})
complete_pairs = sbp_wide_both[(sbp_wide_both["baseline"].notna() == 1) & (sbp_wide_both["week12"].notna() == 1)]
complete_pairs = np.array(complete_pairs, dtype = float)

# Displays shape and column means for the complete pairs table.
display(complete_pairs.shape)
complete_pairs_means = complete_pairs.mean(axis = 0)
display(complete_pairs_means)

(27149, 2)

array([143.01337066, 137.15061328])

### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = pd.merge(visits_clean, patients_clean, on = ["patient_id"], how = "left", validate = "many_to_one")
analysis = pd.merge(analysis, sites_current, on = ["site_code"], how = "left", validate = "many_to_one")
analysis = pd.merge(analysis, a1c_long, on = ["patient_id", "visit"], how = "left", validate = "many_to_one")

# Updates analysis with the dtypes specified in task 4.4.
analysis = analysis.astype({"patient_id": "string", "visit": "string", "visit_date": "datetime64[us]", "sbp_mmhg": "Int64", "weight_kg": "float64", "a1c_pct": "float64", "gender": "string", "birth_year": "Int64", "site_code": "string", "site_name": "string", "source_file": "string"})
display(analysis)

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,source_file,gender,birth_year,site_code,site_name,record_status,a1c_pct
0,P00001,baseline,2026-03-06,146,52.4,visits_south.csv,F,1966,S02,South Valley Clinic,current,7.3
1,P00001,week12,2026-05-27,129,50.2,visits_south.csv,F,1966,S02,South Valley Clinic,current,NaN
2,P00002,baseline,2026-02-20,132,85.2,visits_north.csv,F,<NA>,N01,North Clinic,current,8.4
3,P00002,week12,2026-05-08,129,82.0,visits_north.csv,F,<NA>,N01,North Clinic,current,7.4
4,P00003,baseline,2026-01-16,148,67.5,visits_north.csv,F,1970,N01,North Clinic,current,5.9
...,...,...,...,...,...,...,...,...,...,...,...,...
57176,P30038,week12,2026-04-20,135,109.0,visits_south.csv,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
57177,P30039,baseline,2026-02-24,142,81.8,visits_south.csv,<NA>,<NA>,<NA>,<NA>,<NA>,8.0
57178,P30039,week12,2026-05-15,138,80.3,visits_south.csv,<NA>,<NA>,<NA>,<NA>,<NA>,7.6
57179,P30040,baseline,2026-02-04,144,121.0,visits_south.csv,<NA>,<NA>,<NA>,<NA>,<NA>,NaN


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

# Checks if raw and raw_snapshot match for each table, making sure there are no discrepencies.
x = 0
checker = 0 
display(raw.keys())
for keys in raw.keys():
    true_key = (((raw[keys] != raw_snapshot[keys])).sum(axis=0).all()==0)
    if true_key == True:
        x = x + 1
    if x == 5:
        checker = 1

# Created a series to check the requirements for task 5.1.
checks = pd.Series( {
    "analysis has as many rows as visits_clean": analysis.shape[0] == visits_clean.shape[0],
    "no patient_id and visit pair repeats in analysis": sum(analysis.duplicated(subset = ["patient_id", "visit"], keep=False)) == 0,
    "every sbp_mmhg that is present in analysis is from 70 through 250": (~(analysis["sbp_mmhg"].between(70, 250))).sum()==0,
    "every site_name that is present is a current site name": (~(analysis["record_status"]=="current")).sum()==0,
    "sbp_wide[patient_id] is unique": sum(sbp_wide.duplicated(subset = ["patient_id"], keep=False)) == 0,
    "every raw table still equals its snapshot": checker == 1
    })

display(checks)
assert checks.all(), checks[~checks]


dict_keys(['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide'])

analysis has as many rows as visits_clean                            True
no patient_id and visit pair repeats in analysis                     True
every sbp_mmhg that is present in analysis is from 70 through 250    True
every site_name that is present is a current site name               True
sbp_wide[patient_id] is unique                                       True
every raw table still equals its snapshot                            True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

# Validation step to check that analysis_sample has the correct number of rows.
analysis_sample.shape[0] == visits_clean["patient_id"].isin(SAMPLE_IDS).sum()

np.True_

### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.
# Checked dtypes for analysis_sample before reading back.
analysis_sample_types = analysis_sample.dtypes.drop("visit_date")
analysis_sample_types = analysis_sample.dtypes
analysis_sample_types

# Reading back analysis_sample.
round_trip = pd.read_csv(ANALYSIS_SAMPLE_PATH, dtype={"patient_id": "string", "visit": "string", "sbp_mmhg": "Int64",
                                                     "weight_kg": "float64", "source_file": "string", "gender": "string",
                                                     "birth_year": "Int64", "site_code": "string", "site_name": "string", "record_status": "string",
                                                     "a1c_pct": "float64"}, parse_dates=["visit_date"])

round_trip.equals(analysis_sample)

True